# paintingReorganize — GPU iteration

Rearranges a painting's pixels into a smooth palette. Same pixels, same
dense rectangle.

**First: Runtime → Change runtime type → T4 GPU.** Everything below assumes one.

On CPU a full-resolution Starry Night takes ~95 min; on a T4 expect a few
minutes, which is what makes parameter sweeps practical.


In [ ]:
!nvidia-smi -L || echo "NO GPU - use Runtime > Change runtime type > T4 GPU"
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Get the code and the paintings

In [ ]:
!git clone -q --branch claude/brave-newton-xsqgpe https://github.com/ardila/paintingReorganize.git repo || (cd repo && git pull -q)
%cd repo
!pip -q install scipy pillow
import numpy as np, torch, time
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import smooth_palette_gpu as G
print("ready")

## Benchmark

Measures ms/sweep so you can predict any run's cost before starting it.
CPU reference on this workload was ~250 ms/sweep at 0.9 MP.

In [ ]:
rgb = np.asarray(Image.open('demoiselles.jpg').convert('RGB'))
t = time.time(); out = G.run(rgb, sweeps=200, device='cuda', log_every=100); el = time.time()-t
mp = rgb.shape[0]*rgb.shape[1]/1e6
print(f"{mp:.2f} MP -> {el/200*1000:.1f} ms/sweep; a 6000-sweep run = {el/200*6000/60:.1f} min")

## Parameter sweep

This is the loop worth having: several configs, rendered side by side.

- `top_frac` — widest kernel sigma as a fraction of the short side. This
  controls how far a large colour region can pull a small one. Too small
  and the picture keeps mid-scale blotches; it is why fixed-pixel kernels
  failed on big images.
- `lam` — composition-field strength. 0 collapses to a bullseye; too high
  over-constrains and pinches colour regions apart. ~12 worked at 0.9 MP.


In [ ]:
import matplotlib.pyplot as plt

SRC     = 'demoiselles.jpg'   # or starry_night.png, the_large_bathers.jpg, input.jpg
SCALE   = 0.5                 # downscale for fast sweeps; 1.0 = full res
SWEEPS  = 2000
CONFIGS = [dict(top_frac=f, lam=l) for f in (0.02, 0.06, 0.15) for l in (12.0,)]

im = Image.open(SRC).convert('RGB')
if SCALE != 1.0:
    im = im.resize((int(im.width*SCALE), int(im.height*SCALE)), Image.LANCZOS)
rgb = np.asarray(im)
print(f"{rgb.shape[1]}x{rgb.shape[0]}")

results = []
for cfg in CONFIGS:
    t = time.time()
    out = G.run(rgb, sweeps=SWEEPS, device='cuda', verbose=False, **cfg)
    results.append((cfg, out, time.time()-t))
    print(f"{cfg} -> {time.time()-t:.0f}s")

n = len(results)+1
fig, ax = plt.subplots(1, n, figsize=(5*n, 5))
ax[0].imshow(rgb); ax[0].set_title('original'); ax[0].axis('off')
for i, (cfg, out, el) in enumerate(results, start=1):
    ax[i].imshow(out); ax[i].axis('off')
    ax[i].set_title(f"top_frac={cfg['top_frac']} lam={cfg['lam']}\n{el:.0f}s")
plt.tight_layout(); plt.show()

## Zoom in

Sweep-sized previews hide grain. Always check a 1:1 crop before believing a result.

In [ ]:
cfg, out, _ = results[-1]
y, x = out.shape[0]//2, out.shape[1]//2
plt.figure(figsize=(9,9)); plt.imshow(out[y-200:y+200, x-200:x+200])
plt.title(f'1:1 crop  {cfg}'); plt.axis('off'); plt.show()

## Full-quality run

Note the annealing gets **worse before better** — roughness rises to ~2.5x
its starting value at peak heat before falling below it. Runs under ~2500
sweeps land in the damaged phase and look like regressions.

In [ ]:
SRC = 'starry_night.png'
rgb = np.asarray(Image.open(SRC).convert('RGB'))
t = time.time()
out = G.run(rgb, sweeps=6000, lam=12.0, top_frac=0.06, device='cuda')
print(f"{(time.time()-t)/60:.1f} min")
Image.fromarray(out).save('result.png')
plt.figure(figsize=(16,10)); plt.imshow(out); plt.axis('off'); plt.show()
from google.colab import files; files.download('result.png')

## Garden with Peacocks (10.6 MP)

Hopeless on CPU (~5 h) and the reason the GPU port exists.

In [ ]:
!wget -q -O peacocks.jpg "https://commons.wikimedia.org/wiki/Special:FilePath/Franti%C5%A1ek_Kupka_%E2%80%93_Garden_with_Peacocks.jpg?width=3609"
rgb = np.asarray(Image.open('peacocks.jpg').convert('RGB'))
print(rgb.shape)
t = time.time()
out = G.run(rgb, sweeps=6000, lam=12.0, top_frac=0.06, device='cuda')
print(f"{(time.time()-t)/60:.1f} min")
Image.fromarray(out).save('peacocks_smooth.png')
plt.figure(figsize=(16,13)); plt.imshow(out); plt.axis('off'); plt.show()